# RAG Query Engine and Chatbot Memory

Your current notebook can already retrieve relevant chunks from ChromaDB. The next stage is to turn retrieval into a chatbot pipeline.

The idea is:

1. Keep your hybrid retriever: semantic search plus BM25 keyword search, fused with reciprocal rank fusion.
2. Wrap that retriever in a LlamaIndex `QueryEngine` for single-turn question answering.
3. Wrap the same retriever in a LlamaIndex `ChatEngine` for multi-turn chatbot conversations.
4. Create one memory object per chat, so different chats remember different conversation histories.

This is close to how a production chatbot is organized: retrieval finds evidence, the LLM writes the answer, and chat memory preserves the local conversation context.

In [ ]:
# Install the extra LlamaIndex packages needed for BM25 and Ollama-backed generation.
# Run this once, then restart the kernel if a new import is not recognized.
%pip install -U llama-index-retrievers-bm25 llama-index-llms-ollama ollama

In [1]:
from collections import defaultdict
from pathlib import Path
from typing import Any

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import QueryBundle, Settings, VectorStoreIndex
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore

resource module not available on Windows


In [2]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
PDF_DIR = PROJECT_ROOT / "data" / "wikipedia"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "mini_wikipedia"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
OLLAMA_MODEL = "gemma3:1b"

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_DIR}")

# LlamaIndex uses Settings as global defaults for retrieval and generation.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = Ollama(model=OLLAMA_MODEL, request_timeout=120.0)

print(f"PDF source folder: {PDF_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM model: {OLLAMA_MODEL}")

PDF source folder: C:\Program Files\Studying\coding\RAG_project\data\wikipedia
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Embedding model: BAAI/bge-small-en-v1.5
LLM model: gemma3:1b


In [3]:
def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


In [4]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def load_nodes_from_chroma_collection(collection) -> list[TextNode]:
    """Rebuild TextNode objects from the persisted Chroma collection for keyword retrieval."""
    # Chroma stores text and metadata, so we can reconstruct nodes without rerunning ingestion.
    records = collection.get(include=["documents", "metadatas"])
    documents = records.get("documents", []) or []
    metadatas = records.get("metadatas", []) or []
    ids = records.get("ids", []) or []

    nodes: list[TextNode] = []
    for node_id, document_text, metadata in zip(ids, documents, metadatas):
        if not document_text:
            continue
        nodes.append(
            TextNode(
                id_=node_id,
                text=document_text,
                metadata=metadata or {},
            )
        )

    if not nodes:
        raise ValueError("No stored text nodes were found in the Chroma collection.")

    return nodes


def build_bm25_retriever(collection, top_k: int = 5) -> BM25Retriever:
    """Create a BM25 keyword retriever from the persisted Chroma text and metadata."""
    # BM25 complements embeddings by rewarding exact keyword overlap.
    nodes = load_nodes_from_chroma_collection(collection)
    return BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)


def reciprocal_rank_fusion(
    semantic_results: list[NodeWithScore],
    keyword_results: list[NodeWithScore],
    top_k: int,
    rrf_k: int = 60,
) -> list[NodeWithScore]:
    """Fuse semantic and keyword rankings using reciprocal rank fusion."""
    # RRF combines rankings instead of raw scores, which makes it robust across retrievers.
    fused_scores: dict[str, float] = defaultdict(float)
    node_lookup: dict[str, NodeWithScore] = {}

    for results in (semantic_results, keyword_results):
        for rank, result in enumerate(results, start=1):
            node_id = result.node.node_id
            fused_scores[node_id] += 1.0 / (rrf_k + rank)

            # Keep the first full node payload so we can print the final fused results later.
            if node_id not in node_lookup:
                node_lookup[node_id] = result

    reranked_results = sorted(
        node_lookup.values(),
        key=lambda result: fused_scores[result.node.node_id],
        reverse=True,
    )

    fused_results: list[NodeWithScore] = []
    for result in reranked_results[:top_k]:
        node_id = result.node.node_id
        fused_results.append(NodeWithScore(node=result.node, score=fused_scores[node_id]))

    return fused_results


def print_ranked_results(results: list[NodeWithScore], title: str) -> None:
    """Print retrieval results with source metadata so ranking quality is easy to inspect."""
    print(f"\n{title}")
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
    collection=None,
    bm25_retriever: BM25Retriever | None = None,
    rrf_k: int = 60,
):
    """Run semantic search, keyword search, then fuse both rankings with reciprocal rank fusion."""
    # Load the stored vector index lazily so the function still works after reopening the notebook.
    active_index = query_index if query_index is not None else load_existing_index()

    # Semantic retrieval is strong for meaning and paraphrase matches.
    semantic_retriever = active_index.as_retriever(similarity_top_k=top_k)
    semantic_results = semantic_retriever.retrieve(query)

    # BM25 retrieval is strong for exact terms, names, acronyms, and rare keywords.
    active_collection = collection if collection is not None else get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    active_bm25_retriever = bm25_retriever if bm25_retriever is not None else build_bm25_retriever(active_collection, top_k=top_k)
    keyword_results = active_bm25_retriever.retrieve(query)

    # Reciprocal rank fusion combines the strengths of both rankings into one final list.
    fused_results = reciprocal_rank_fusion(semantic_results, keyword_results, top_k=top_k, rrf_k=rrf_k)

    # print_ranked_results(semantic_results, "Semantic Search Results")
    # print_ranked_results(keyword_results, "BM25 Keyword Search Results")
    print_ranked_results(fused_results, "Hybrid Search Results (RRF)")

    return {
        "semantic_results": semantic_results,
        "keyword_results": keyword_results,
        "hybrid_results": fused_results,
    }

## Build a LlamaIndex Query Engine and Chat Engine

A `QueryEngine` is for single-turn RAG: the user asks one question, the retriever finds context, and the LLM writes one answer.

A `ChatEngine` is the next step for a chatbot: it still retrieves context from your vector database, but it also keeps chat memory so follow-up questions can refer to earlier messages.

The design below keeps one shared knowledge base in ChromaDB, but creates separate memory for each chat. That means Chat A and Chat B can talk about different things without mixing their histories.

In [5]:
class HybridRetriever(BaseRetriever):
    """LlamaIndex retriever that fuses semantic search and BM25 keyword search."""

    def __init__(
        self,
        semantic_retriever,
        keyword_retriever: BM25Retriever,
        final_top_k: int = 5,
        rrf_k: int = 60,
    ):
        # Keep the two retrievers separate so we can inspect or tune them independently.
        self._semantic_retriever = semantic_retriever
        self._keyword_retriever = keyword_retriever
        self._final_top_k = final_top_k
        self._rrf_k = rrf_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> list[NodeWithScore]:
        """Retrieve with both methods, then return one fused ranking."""
        # Semantic search finds conceptually similar chunks even when the wording is different.
        semantic_results = self._semantic_retriever.retrieve(query_bundle)

        # BM25 keyword search protects exact terms such as names, dates, acronyms, and places.
        keyword_results = self._keyword_retriever.retrieve(query_bundle)

        # RRF merges ranks without requiring semantic and BM25 scores to be on the same scale.
        return reciprocal_rank_fusion(
            semantic_results,
            keyword_results,
            top_k=self._final_top_k,
            rrf_k=self._rrf_k,
        )


def build_hybrid_retriever(
    query_index: VectorStoreIndex,
    collection,
    final_top_k: int = 5,
    candidate_top_k: int | None = None,
    rrf_k: int = 60,
) -> HybridRetriever:
    """Build the shared retriever used by both the query engine and chat engine."""
    # Pull a slightly larger candidate set before fusion so RRF has room to improve ranking.
    candidate_count = candidate_top_k if candidate_top_k is not None else max(final_top_k * 2, 10)

    semantic_retriever = query_index.as_retriever(similarity_top_k=candidate_count)
    keyword_retriever = build_bm25_retriever(collection, top_k=candidate_count)

    return HybridRetriever(
        semantic_retriever=semantic_retriever,
        keyword_retriever=keyword_retriever,
        final_top_k=final_top_k,
        rrf_k=rrf_k,
    )


def build_rag_query_engine(
    hybrid_retriever: HybridRetriever,
    llm=None,
) -> RetrieverQueryEngine:
    """Create a single-turn RAG query engine from the hybrid retriever."""
    # QueryEngine is best for isolated questions where chat memory is not needed.
    active_llm = llm if llm is not None else Settings.llm
    return RetrieverQueryEngine.from_args(retriever=hybrid_retriever, llm=active_llm)


CHAT_SYSTEM_PROMPT = """
You are a helpful RAG chatbot for a mini-Wikipedia knowledge base.
Use the retrieved context as your primary evidence.
If the retrieved context does not contain enough information, say that the knowledge base does not have enough evidence.
Use the current chat history to understand follow-up questions, but do not invent facts that are not supported by the retrieved context.
""".strip()


def build_rag_chat_engine(
    hybrid_retriever: HybridRetriever,
    memory: ChatMemoryBuffer,
    llm=None,
) -> ContextChatEngine:
    """Create a multi-turn chat engine that combines retrieval with chat memory."""
    # ChatEngine adds conversation memory around the same retriever used by the query engine.
    active_llm = llm if llm is not None else Settings.llm
    return ContextChatEngine.from_defaults(
        retriever=hybrid_retriever,
        memory=memory,
        llm=active_llm,
        system_prompt=CHAT_SYSTEM_PROMPT,
    )


chat_sessions: dict[str, dict[str, Any]] = {}


def create_chat_session(
    chat_id: str,
    hybrid_retriever: HybridRetriever,
    memory_token_limit: int = 3000,
) -> str:
    """Create one independent chat memory and chat engine for a conversation."""
    # Each chat gets its own memory object, so different chats do not share history.
    memory = ChatMemoryBuffer.from_defaults(token_limit=memory_token_limit)
    chat_engine = build_rag_chat_engine(hybrid_retriever=hybrid_retriever, memory=memory)

    chat_sessions[chat_id] = {
        "memory": memory,
        "chat_engine": chat_engine,
    }
    return chat_id


def chat_with_rag(chat_id: str, message: str):
    """Send a message to one chat session and print the model answer."""
    if chat_id not in chat_sessions:
        raise KeyError(f"Chat session not found: {chat_id}. Create it with create_chat_session first.")

    # The chat engine uses both retrieved context and this chat's previous messages.
    response = chat_sessions[chat_id]["chat_engine"].chat(message)
    print(response.response)
    return response


def show_chat_history(chat_id: str) -> None:
    """Print the stored messages for one chat session."""
    if chat_id not in chat_sessions:
        raise KeyError(f"Chat session not found: {chat_id}")

    # ChatMemoryBuffer stores the user and assistant turns that belong to this chat only.
    messages = chat_sessions[chat_id]["memory"].get_all()
    for message in messages:
        print(f"{message.role}: {message.content}")


def print_response_sources(response, max_sources: int = 3) -> None:
    """Print source chunks used by a query or chat response."""
    # Source nodes help you debug whether the answer is grounded in the right retrieved text.
    source_nodes = getattr(response, "source_nodes", []) or []
    for rank, source_node in enumerate(source_nodes[:max_sources], start=1):
        metadata = source_node.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        print(f"\nSource {rank}: {file_name}, page {page_label}")
        print(source_node.node.get_content()[:800])

## Run the RAG Engines

Run the next cell once per notebook session. It reconnects to ChromaDB, rebuilds the hybrid retriever, and creates a query engine.

Use the query engine for one-off questions. Use the chat engine when the user is in a conversation and may ask follow-up questions like "why?", "summarize that", or "compare it with the first one".

In [7]:
# Run this once after reopening the notebook to reconnect to persisted storage.
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
reloaded_index = load_existing_index()

# Build one shared hybrid retriever and query engine for this notebook session.
hybrid_retriever = build_hybrid_retriever(
    query_index=reloaded_index,
    collection=collection,
    final_top_k=5,
)

In [8]:
# Single-turn RAG: use the query engine when the question does not need chat history.
rag_query_engine = build_rag_query_engine(hybrid_retriever)

print(f"Stored vector count: {collection.count()}")
print("Hybrid retriever and RAG query engine are ready.")

sample_query = "Which university is the best in Hong Kong?"
query_response = rag_query_engine.query(sample_query)

print(query_response)
print('-' * 70)
print_response_sources(query_response)

Stored vector count: 836
Hybrid retriever and RAG query engine are ready.
According to the provided text, the University of Hong Kong (HKU) is ranked as the best university in Hong Kong.
----------------------------------------------------------------------

Source 1: Hong Kong.pdf, page 40
University (in 2006), Education University of Hong Kong (in 2016), Hang Seng University of Hong
Kong (in 2018) and Saint Francis University (in 2024) all attained full university status in subsequent
years.
Overseas universities in Hong Kong often operate as branch campuses or partnerships offering
accredited degrees, such as the University of Chicago Hong Kong, University of Sunderland,
University of Wollongong in Hong Kong. These institutions, provide local and internationally
recognized qualifications.
In 2026, QS Best Student Cities ranked Hong Kong as the 17th best city for university students. 7th
in best Asia for best student cities. Noting, high scores in employer activity, desirability, and

In [9]:
# Multi-turn RAG: create different chat sessions for different conversations.
university_chat_id = create_chat_session("hong_kong_university_chat", hybrid_retriever)

chat_response = chat_with_rag(
    chat_id = university_chat_id,
    message = "Which university is the best in Hong Kong?",
)
print_response_sources(chat_response)

According to the provided context, the University of Hong Kong (HKU) is ranked #1 in QS World University Rankings 2026, #1 in Times Higher Education World University Rankings 2025, and #1 in the Academic Ranking of World Universities 2024.

Source 1: Hong Kong.pdf, page 40
University (in 2006), Education University of Hong Kong (in 2016), Hang Seng University of Hong
Kong (in 2018) and Saint Francis University (in 2024) all attained full university status in subsequent
years.
Overseas universities in Hong Kong often operate as branch campuses or partnerships offering
accredited degrees, such as the University of Chicago Hong Kong, University of Sunderland,
University of Wollongong in Hong Kong. These institutions, provide local and internationally
recognized qualifications.
In 2026, QS Best Student Cities ranked Hong Kong as the 17th best city for university students. 7th
in best Asia for best student cities. Noting, high scores in employer activity, desirability, and diverse
student p

In [10]:
# Follow-up question: this uses the same chat memory, so "that answer" refers to the prior turn.
follow_up_response = chat_with_rag(
    chat_id = university_chat_id,
    message = "Can you explain the reason for that answer in simpler words?",
)
print_response_sources(follow_up_response)

Okay, here's a simpler explanation of why HKU is considered the best university in Hong Kong:

The text states that **HKU consistently ranks highly in multiple prestigious ranking systems.** Specifically, it’s consistently #1 in:

*   **QS World University Rankings**
*   **Times Higher Education World University Rankings**
*   **Academic Ranking of World Universities**

Basically, it's consistently recognized as the top university in Hong Kong across these major rankings. 



Source 1: Mathematics.pdf, page 17
has yet to be proven (or disproven), it is termed a conjecture. Through a series of rigorous
arguments employing deductive reasoning, a statement that is proven to be true becomes a
theorem. A specialized theorem that is mainly used to prove another theorem is called a lemma. A
proven instance that forms part of a more general finding is termed a corollary.
Numerous technical terms used in mathematics are neologisms, such as polynomial and
homeomorphism. Other technical terms are

In [13]:
# Inspect memory for one chat. A different chat_id has a different history.
show_chat_history(chat_id = university_chat_id)

MessageRole.USER: Which university is the best in Hong Kong?
MessageRole.ASSISTANT: According to the provided context, the University of Hong Kong (HKU) is ranked #1 in QS World University Rankings 2026, #1 in Times Higher Education World University Rankings 2025, and #1 in the Academic Ranking of World Universities 2024.
MessageRole.USER: Can you explain the reason for that answer in simpler words?
MessageRole.ASSISTANT: Okay, here's a simpler explanation of why HKU is considered the best university in Hong Kong:

The text states that **HKU consistently ranks highly in multiple prestigious ranking systems.** Specifically, it’s consistently #1 in:

*   **QS World University Rankings**
*   **Times Higher Education World University Rankings**
*   **Academic Ranking of World Universities**

Basically, it's consistently recognized as the top university in Hong Kong across these major rankings. 


